# Supabase Storage - Arquivos

Documentação da integração do BetterAI com o Supabase Storage: como configurar buckets e enviar, ler, listar e remover arquivos, inclusive junto com um registro de tabela.

### Visão Geral

A gestão de arquivos nos buckets do Supabase é feita pela mesma integração das tabelas, em `src/database/supabase_integration/`. Os fluxos do projeto não usam o SDK `supabase` diretamente: pedem a integração e usam `sb.storage`.

- **`get_supabase().storage`**: a classe de arquivos (`FileStore`) com `upload`, `download`, `download_to`, `signed_url`, `public_url`, `exists`, `list`, `delete` e `delete_prefix`.
- **`BytesSource`, `PathSource`, `StreamSource`**: as três formas de entregar o conteúdo a enviar.
- **`UploadOptions`**: bucket, pasta, caminho explícito, política de colisão, tipo de conteúdo e metadados.
- **`StorageRepository`** (`src/storage/storage_repository.py`): wrapper do projeto para enviar imagens e obter a URL pública, usado pela geração de imagens.

> Todas as operações validam a entrada antes de ir à rede (bucket permitido, tamanho, extensão, tipo declarado e conteúdo real) e levantam exceções do domínio em vez de devolver `None`.

### Fluxo de Upload

1. **Bucket:** `UploadOptions.bucket` (ou `SBX_BUCKET_DEFAULT`) é resolvido e conferido contra a allowlist de buckets.
2. **Validação:** tamanho (vazio e acima do teto são recusados), extensão, tipo declarado e conteúdo real (magic bytes) precisam concordar.
3. **Caminho:** `path` explícito, ou gerado como `<folder>/AAAA/MM/<uuid><ext>`.
4. **Envio:** upload padrão ou, acima de `SBX_STORAGE_RESUMABLE_THRESHOLD_BYTES` (6 MiB), upload resumable (TUS), lido em stream.
5. **Colisão:** se o caminho já existe, vale a `CollisionPolicy` (`REJECT`, `VERSION` ou `OVERWRITE`).
6. **Resultado:** um `StoredFile` com `bucket`, `path`, `size`, `content_type`, `original_name`, `upload_mode` e `metadata`.

### Configuração

A configuração vem das variáveis de ambiente `SBX_*`. As usadas pelos arquivos são:

| Variável | Descrição |
|---|---|
| `SBX_URL` | URL https do projeto Supabase (obrigatória) |
| `SBX_SECRET_KEY` | Chave secret do backend (obrigatória; nunca no frontend) |
| `SBX_BUCKET_DEFAULT` | Bucket padrão (ex.: `images`) |
| `SBX_BUCKETS_ALLOWLIST` | Buckets permitidos, separados por vírgula (obrigatória em prod) |
| `SBX_BUCKETS_PUBLIC` | Buckets públicos; só neles `public_url` funciona |
| `SBX_STORAGE_MAX_FILE_BYTES` | Tamanho máximo do arquivo (50 MiB) |
| `SBX_STORAGE_ALLOWED_MIME` | Tipos MIME permitidos; aceita `image/*`; vazio aceita qualquer |
| `SBX_STORAGE_SIGNED_URL_TTL_SECONDS` | Validade padrão da URL assinada (300) |
| `SBX_STORAGE_SIGNED_URL_MAX_TTL_SECONDS` | Teto da validade da URL assinada (3600) |
| `SBX_STORAGE_RESUMABLE_THRESHOLD_BYTES` | Tamanho a partir do qual o upload vira resumable (6 MiB) |
| `SBX_STORAGE_MEMORY_CAP_BYTES` | Teto de memória para `BytesSource` e `download` (32 MiB) |
| `SBX_STORAGE_UPLOAD_TIMEOUT_SECONDS` | Timeout do upload em segundos (120) |
| `SBX_STORAGE_MAX_CONCURRENT_TRANSFERS` | Transferências simultâneas (4) |

> **Bucket público.** Para gerar a URL pública das imagens, o bucket precisa constar em `SBX_BUCKETS_PUBLIC` (por exemplo `SBX_BUCKET_DEFAULT=images`, `SBX_BUCKETS_PUBLIC=images`, `SBX_BUCKETS_ALLOWLIST=images`). Em `homol` e `prod` o `.env` de arquivo é proibido. A lista completa está em `src/database/supabase_integration/.env.example`.

### Upload

```python
from src.database.supabase_provider import get_supabase
from src.database.supabase_integration import BytesSource, PathSource, StreamSource, UploadOptions, CollisionPolicy

storage = get_supabase().storage

st = storage.upload(
    BytesSource(b"content", filename="note.txt", content_type="text/plain"),
    UploadOptions(folder="docs"),  # Caminho gerado: docs/AAAA/MM/<uuid>.txt
)
print(st.bucket, st.path, st.size, st.upload_mode, st.content_type, st.original_name)

big = storage.upload(
    PathSource("video.mp4", content_type="video/mp4"),  # Lido em stream; acima de 6 MiB vira resumable (TUS)
    UploadOptions(folder="videos"),
)

with open("report.pdf", "rb") as f:
    storage.upload(StreamSource(f, size=None, filename="report.pdf", content_type="application/pdf"))  # Stream, por exemplo o UploadFile do FastAPI

storage.upload(
    BytesSource(b"v2", filename="a.txt", content_type="text/plain"),
    UploadOptions(path=st.path, collision=CollisionPolicy.VERSION),  # Colisão: grava em um caminho novo (-v2)
)
```

#### Fontes de dados

| Classe | Campos | Notas |
|---|---|---|
| `BytesSource` | `data`, `filename`, `content_type` | Em memória; acima de `SBX_STORAGE_MEMORY_CAP_BYTES` levanta `ValidationError` (use `PathSource` ou `StreamSource`). |
| `PathSource` | `path`, `filename`, `content_type` | Lê do disco em stream; `filename` padrão é o nome do arquivo; arquivo inexistente levanta `ValidationError`. |
| `StreamSource` | `stream`, `size`, `filename`, `content_type`, `restartable` | Com `size` e stream que permite `seek`, é relido em caso de retry; sem `size` é copiado para um arquivo temporário (os bytes são contados e o teto é verificado). |

#### UploadOptions

| Campo | Efeito |
|---|---|
| `bucket` | `None` usa `SBX_BUCKET_DEFAULT`; precisa estar na allowlist. |
| `folder` | Subpasta lógica (validada). Caminho gerado: `<folder>/AAAA/MM/<uuid><ext>`. |
| `path` | Caminho explícito, que substitui a nomeação automática e é sempre validado. |
| `collision` | `REJECT` (padrão: `FileAlreadyExistsError`), `VERSION` (grava em `nome-v2.ext`, `-v3`...) ou `OVERWRITE` (sobrescreve, só explícito). |
| `content_type` | Tipo MIME declarado, confrontado com a extensão e o conteúdo. |
| `metadata` | Metadados `str` para `str` (até 50 entradas); a integração acrescenta `original_name`. |
| `cache_control_seconds` | `None` usa `SBX_STORAGE_CACHE_CONTROL_SECONDS`. |
| `correlation_id` | Identificador para amarrar os logs da operação. |

> O tipo do conteúdo é escolhido assim: conteúdo detectado (se específico), depois o declarado, depois a extensão, por fim `application/octet-stream`. Extensão, tipo declarado e conteúdo precisam concordar, senão levanta `ValidationError`. Em bucket público, tipos ativos (`.html`, `.svg`, `.js`, executáveis) são recusados.

> Caminhos de objeto só aceitam ASCII `[A-Za-z0-9._-]`, sem acentos nem espaços, sem segmento começando com ponto e até 8 níveis. No caminho gerado, o nome original do usuário vai como metadado (`original_name`), não no caminho.

### Upload de Imagens no Projeto

A geração de imagens grava os arquivos pelo `StorageRepository`, que usa a integração: monta o caminho `<base_path>/<file_name>`, envia os bytes e devolve a URL pública. Se o upload falha, levanta `RuntimeError` com a causa original em `__cause__`.

```python
from src.storage.storage_repository import StorageRepository

repository = StorageRepository(base_path="image_generations", bucket_name="images")

url = repository.upload_to_supabase(file_name="example_image.png", byte_data=image_bytes)
print(url)  # URL pública do arquivo no bucket
```

### Leitura

```python
data = storage.download(st.path)  # bytes, limitado pelo teto de memória (32 MiB)
storage.download_to(big.path, "out/video.mp4")  # Stream direto para o disco, para arquivos grandes
url = storage.signed_url(st.path, ttl_seconds=300)  # URL assinada; teto 3600 s, acima disso ValidationError
pub = storage.public_url(st.path, bucket="images")  # Só para bucket em SBX_BUCKETS_PUBLIC
found = storage.exists(st.path)
page = storage.list(prefix="docs", limit=100)  # Page[ObjectInfo]
for o in page.items:
    print(o.path, o.size, o.content_type, o.updated_at, o.is_folder)
```

> `exists` devolve `False` também em 401 e 403 (limitação do SDK): um problema de permissão pode parecer "não existe".

### Remoção

```python
rep = storage.delete([st.path, big.path])  # Idempotente; relatório por caminho
print(rep.requested, rep.removed, rep.failed)  # failed: {caminho: motivo}
storage.delete_prefix("docs/2026/11")  # DESTRUTIVO: apaga tudo sob o prefixo, recursivamente
```

> Falhas parciais em `delete` não levantam exceção: confira `rep.failed`. `delete_prefix` é destrutivo e recursivo (profundidade máxima 16); prefixo vazio é proibido.

### Tabela de Métodos

| Método | Retorno | Descrição Breve |
|---|---|---|
| `upload(source, options=None)` | `StoredFile` | Valida e envia o arquivo; `upload_mode` é `standard` ou `resumable`. |
| `download(path, bucket=None)` | `bytes` | Baixa para a memória; recusa objeto acima do teto (`use download_to`). |
| `download_to(path, destination, bucket=None)` | `Path` | Baixa em stream para o disco e cria as pastas pai. |
| `signed_url(path, bucket=None, ttl_seconds=None)` | `str` | URL temporária; nunca é registrada em log. |
| `public_url(path, bucket=None)` | `str` | URL pública, só para bucket em `SBX_BUCKETS_PUBLIC`; montada localmente, sem rede. |
| `exists(path, bucket=None)` | `bool` | Indica se o objeto existe. |
| `list(prefix="", bucket=None, limit=None, offset=0)` | `Page[ObjectInfo]` | Lista objetos; pastas vêm com `is_folder=True`. |
| `delete(paths, bucket=None)` | `DeleteReport` | Remove em lotes; idempotente; falhas vão para `failed`. |
| `delete_prefix(prefix, bucket=None)` | `DeleteReport` | Remove tudo sob o prefixo, recursivamente. |

### Arquivo e Registro Juntos

Quando o arquivo precisa ser gravado junto com um registro de tabela, use as operações combinadas da fachada. Elas seguem uma ordem fixa e compensam em caso de falha: se o registro falha depois do upload, o arquivo é apagado. As tabelas são descritas na página de Supabase Database.

```python
sb = get_supabase()

r = sb.save_file_with_record(
    "sbx_test_items",
    BytesSource(b"content", filename="contract.txt", content_type="text/plain"),
    lambda f: {"name": "contract", "bucket": f.bucket, "path": f.path},  # Monta o registro a partir do arquivo gravado
    UploadOptions(folder="contracts"),
)
r.raise_if_failed()  # Levanta o mesmo erro original, se houve falha
print(r.status.value, r.record["id"])  # completa, parcial ou falhou

sb.replace_file_in_record(
    "sbx_test_items", r.record["id"],
    BytesSource(b"v2", filename="contract.txt", content_type="text/plain"),
    lambda f: {"bucket": f.bucket, "path": f.path},
    old_ref=(r.stored_file.bucket, r.stored_file.path),  # O arquivo antigo é apagado por último
)

sb.delete_record_and_file("sbx_test_items", r.record["id"], (r.stored_file.bucket, r.stored_file.path))  # Registro primeiro, arquivo depois
```

| `status` | Significado | `orphan_possible` |
|---|---|---|
| `completa` | Todos os passos foram concluídos. | `False` |
| `falhou` | Nada ficou para trás (falha antes de gravar, ou compensação bem-sucedida). | `False` |
| `parcial` | Sobrou algo: arquivo órfão ou arquivo antigo não apagado. O log traz `orfao_possivel=true` com bucket e path. | `True` |

> `raise_if_failed()` levanta também no status `parcial`, mesmo quando o registro foi atualizado. Leia `r.status` antes de decidir.

### Tratamento de Erros

| Exceção | Quando |
|---|---|
| `ValidationError` | Bucket fora da allowlist, arquivo vazio, tipo divergente ou não permitido, caminho inválido, URL assinada acima do teto, bucket não público em `public_url`. |
| `FileTooLargeError` | Arquivo acima de `SBX_STORAGE_MAX_FILE_BYTES` ou resposta 413 do Storage. |
| `FileAlreadyExistsError` | O caminho já existe com `CollisionPolicy.REJECT`. |
| `FileNotFoundInStorageError` | Objeto inexistente (o Storage também usa 404 para "sem permissão"). |
| `PermissionDeniedError`, `AuthenticationError` | Política do bucket negou o acesso; chave inválida. |
| `UploadError`, `DownloadError` | Falha de storage não classificada. |
| `OperationTimeoutError`, `ServiceUnavailableError`, `RateLimitError` | Falhas transitórias (`retryable=True`). |

```python
try:
    storage.upload(BytesSource(data, filename="a.png"), UploadOptions(path="images/a.png"))
except FileAlreadyExistsError:
    ...  # O caminho já existe
except SupabaseIntegrationError as e:  # Qualquer erro da integração
    print(e.retryable, e.outcome_unknown, e.correlation_id)
```

### Arquitetura e Insights

- **Upload resumable:** acima de `SBX_STORAGE_RESUMABLE_THRESHOLD_BYTES` o envio usa o protocolo TUS em chunks, sem carregar o arquivo inteiro em memória.
- **Retry seguro:** só `OVERWRITE` é repetido pelo executor. Em `REJECT` e `VERSION` um timeout vira `outcome_unknown=True` e o objeto pode ter sido gravado: confira com `exists`.
- **Nomes seguros:** o caminho gerado nunca usa o nome do usuário, o que evita colisão e caracteres inválidos.
- **Sem bloquear o event loop:** em FastAPI use `create_async_integration`; `await asb.upload(...)` roda em thread.
- **Segredos protegidos:** a chave `SBX_SECRET_KEY` e as URLs assinadas nunca aparecem em logs ou mensagens de erro.